##### Carga de librerías

In [ ]:
import os
import cv2            
import torch          
from ultralytics import YOLO  
import matplotlib.pyplot as plt 

##### Carga del modelo base

In [9]:
print("Cargando el modelo base de YOLO11 Nano...")

model = YOLO("Modelos/yolo11n.pt") 

print("¡Éxito! YOLO11n se ha cargado correctamente en la memoria.")
print(model.info())

Cargando el modelo base de YOLO11 Nano...
¡Éxito! YOLO11n se ha cargado correctamente en la memoria.
YOLO11n summary: 181 layers, 2,624,080 parameters, 0 gradients, 6.6 GFLOPs
(181, 2624080, 0, 6.614336)


##### Entrenamiento del modelo base

In [ ]:
results = model.train(
    data="Dataset_Frutas_Final_Red.yolov11/data.yaml",
    epochs=50, 
    imgsz=512, 
    batch=8, 
    device="cuda"  # Cambia a "cuda" si tienes GPU Nvidia
)

New https://pypi.org/project/ultralytics/8.4.62 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.56  Python-3.12.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Dataset_Frutas_Final_Red.yolov11/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, 

In [ ]:
# Para entrenar con GPU
import torch
print(f"¿Es CUDA disponible?: {torch.cuda.is_available()}")
print(f"Versión de PyTorch con CUDA: {torch.version.cuda}")

¿Es CUDA disponible?: True
Versión de PyTorch con CUDA: 12.1


##### Carga del modelo entrenado

In [4]:
print("Cargando el modelo FruitQual...")

modelo_fruitqual = YOLO('Modelos/best.pt') 

print("¡Éxito! FruitQual se ha cargado correctamente en la memoria.")
print(modelo_fruitqual.info())

Cargando el modelo FruitQual...
¡Éxito! FruitQual se ha cargado correctamente en la memoria.
YOLO11n summary: 182 layers, 2,591,010 parameters, 0 gradients, 6.4 GFLOPs
(182, 2591010, 0, 6.445977599999999)


##### Prueba con Yolo11n

In [10]:
ruta_imagen_prueba = "Pruebas/manzana.jpg"
print("Haciendo una predicción de prueba...")
resultados = model(ruta_imagen_prueba)

for r in resultados:
    r.save(filename="resultado_prueba.jpg")

print("¡Listo! Se ha creado el archivo 'resultado_prueba.jpg' en tu proyecto. Ábrelo para ver las cajas.")

Haciendo una predicción de prueba...

image 1/1 c:\Users\deiko\REPOVISION\Proyecto-vison-borrador\Pruebas\manzana.jpg: 384x640 3 bowls, 5 apples, 54.5ms
Speed: 3.0ms preprocess, 54.5ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
¡Listo! Se ha creado el archivo 'resultado_prueba.jpg' en tu proyecto. Ábrelo para ver las cajas.


##### Prueba con FruitQual

In [11]:
ruta_imagen_prueba = "Pruebas/manzana.jpg"
print("Haciendo una predicción de prueba...")
resultados = modelo_fruitqual(ruta_imagen_prueba)

# Guardar el resultado con las cajas dibujadas en tu computadora
for r in resultados:
    r.save(filename="resultado_prueba2.jpg")

print("¡Listo! Se ha creado el archivo 'resultado_prueba2.jpg' en tu proyecto. Ábrelo para ver las cajas.")

Haciendo una predicción de prueba...

image 1/1 c:\Users\deiko\REPOVISION\Proyecto-vison-borrador\Pruebas\manzana.jpg: 320x512 1 Apple, 1 Rotten Apple, 27.9ms
Speed: 3.1ms preprocess, 27.9ms inference, 1.6ms postprocess per image at shape (1, 3, 320, 512)
¡Listo! Se ha creado el archivo 'resultado_prueba2.jpg' en tu proyecto. Ábrelo para ver las cajas.


In [ ]:

ruta_modelo_experto = "runs/detect/train/weights/best.pt"

if not os.path.exists(ruta_modelo_experto):
    print(f"Error: No se encuentra el modelo entrenado en '{ruta_modelo_experto}'.")
    print("Asegúrate de haber completado el entrenamiento (Celda 5) primero.")
else:
    print("Cargando tu modelo personalizado experto en frescura de frutas...")
    model = YOLO(ruta_modelo_experto)

    print("Abriendo cámara... Coloca una fruta frente a la pantalla. Presiona 'q' para salir.")
    cap = cv2.VideoCapture(0)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Predicción en tiempo real con 50% de confianza mínima
        results = model(frame, stream=True, conf=0.5)

        for r in results:
            annotated_frame = r.plot()

        cv2.imshow("YOLO11 - Detector de Frescura en Tiempo Real", annotated_frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Cámara cerrada.")

In [ ]:
# 1. Carga tu modelo (asegúrate de poner la ruta correcta a tu .pt)
model = YOLO('Modelos.pt') 

# 2. Realiza la inferencia
results = model('test_frutas.jpg')

# 3. Muestra los resultados
for r in results:
    im_array = r.plot()  # Dibuja las cajas y etiquetas
    cv2.imshow('Deteccion de Frutas', im_array)
    cv2.waitKey(0) # Espera a que presiones una tecla para cerrar
    cv2.destroyAllWindows()